# 进阶教程（八）：LangGraph Functional API（函数式编排）

> 与 StateGraph 并列的第二种编排范式：不画图、不声明 State schema，
> 用 `@entrypoint` + `@task` 两个装饰器直接写"函数式工作流"。

## 本讲内容
1. 概念：Functional API 是什么（为什么需要、两大积木、与 Graph API 对比）
2. `@task`：任务（定义、依赖链、调用 LLM/Agent、异步与 timeout/retry）
3. `@entrypoint`：工作流入口（定义、注入参数、异步版）
4. 短时记忆与状态读写（`previous` / `entrypoint.final`）
5. 持久化、序列化与错误后恢复（含 checkpoint 内部结构）
6. interrupt 与 human-in-the-loop（task 内中断、try/except 陷阱、多中断 ID 映射）
7. 确定性与幂等（重放语义精确表述）
8. 事件流：v3 流式协议（stream.messages / output / interrupts）
9. 实战：用 Functional API 重写 02 讲"多角度评审"（与 Send 对照）
10. 常见坑 / 11. 选型 / 12. FAQ

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [2]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)

# ---- 08 讲附加：v3 流式协议在 langgraph 1.2.10 仍是实验性 API，忽略 Beta 警告 ----
warnings.filterwarnings("ignore", message="The v3 streaming protocol.*")


模型就绪: ChatDeepSeek


## 1. 概念：Functional API 是什么

### 1.1 为什么需要它

Graph API（StateGraph）擅长表达**任意图结构**：环、分支、并行、状态归约——
但代价是样板代码：定义 TypedDict、加节点、连边、声明 reducer……

很多真实工作流其实是**线性流水线**（A → B → C），甚至只是"几个任务串起来"。
Functional API 正是为这类场景而生：**用普通 Python 函数编排工作流**，
没有显式 State schema、没有 add_node/add_edge，代码量减半。

### 1.2 两大积木

| 装饰器 | 角色 | 关键特性 |
|---|---|---|
| `@task` | 原子任务单元 | 可并行调度；结果写入 checkpoint（错误后/恢复时不重跑）；支持 timeout/retry |
| `@entrypoint` | 工作流入口 | 普通函数体即编排逻辑；支持依赖注入（`previous` / config）；可配 checkpointer |

关键规则（后面逐一实测）：
- `@task` **只能在 entrypoint 内（或另一个 task 内）调用**，直接调用会报错
- task 的返回值必须 **JSON 可序列化**（要进 checkpoint）
- entrypoint 编译后**就是一个 Pregel 节点**，与 StateGraph 共享同一底层运行时

### 1.3 Functional API vs Graph API（速览）

| 维度 | Functional API | Graph API (StateGraph) |
|---|---|---|
| 状态 schema | 无显式声明，返回值即状态 | TypedDict + Annotated reducer |
| 结构表达 | 代码顺序 / 列表推导 | 任意图（环、并行、条件边） |
| 样板代码 | 少（两个装饰器） | 多（节点、边、schema） |
| 动态并行 | `[task(x) for x in ...]` + `.result()` | `Send` |
| 可观测性 | get_state / v3 流式 | 完整（get_graph / updates 模式） |
| 适用 | 快速原型、线性流水线 | 复杂流程、循环、需要图级调试 |

两者**可以混用**：entrypoint 内可以 `graph.invoke()`，图的节点也可以是
agent（agent 本身是 CompiledStateGraph）——见第 9 节实战。

## 2. `@task`：任务

### 2.1 定义与执行

`@task` 装饰普通函数。在 entrypoint 内调用 `task(...)` 返回一个 **task handle**，
用 `.result()` 取返回值：

In [2]:

from langgraph.func import entrypoint, task

@task
def double(x: int) -> int:
    return x * 2

@entrypoint()
def workflow(x: int):
    r = double(x).result()   # task handle 的 .result() 拿返回值
    return {"doubled": r}

print(workflow.invoke(21))

{'doubled': 42}


**实测：task 在 entrypoint 外调用会报错**（这是最常见的入门错误）——
task 必须由 entrypoint 的运行时调度：

In [3]:

try:
    double(2)
except Exception as e:
    print(f"[实测] task 在 entrypoint 外调用: {type(e).__name__}")

[实测] task 在 entrypoint 外调用: RuntimeError


### 2.2 何时应该用 task

不是所有代码都要包成 task。task 的意义在于：
- **可重放**：task 结果进 checkpoint，中断恢复/出错重试时不重复执行（省钱、省时）
- **可并行**：多个 task 可并发调度（见第 9 节实战）
- **可重试**：`retry_policy` 自动重试（见 2.5）

经验法则：**任何涉及 LLM 调用、API 请求、IO 的代码都应放进 task**；
纯内存计算（列表推导、字符串拼接）放 entrypoint 内联即可。

### 2.3 task 依赖链

task 的输出可以直接作为下一个 task 的输入，形成依赖链——
这就是"函数式"的流水线：

In [4]:

@task
def add_one(x: int) -> int:
    return x + 1

@task
def add_two(x: int) -> int:
    return x + 2

@entrypoint()
def chain(x: int):
    a = add_one(x).result()
    b = add_two(a).result()   # 依赖链：前一个 task 输出喂给下一个
    return {"result": b}

print(chain.invoke(1))

{'result': 4}


### 2.4 task 内调用 LLM / Agent（已实测）

task 里可以调用任何 Runnable：模型、工具、甚至整个 agent——
agent 本身是 CompiledStateGraph，也是一个 Runnable：

In [3]:

from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_weather(city: str) -> str:
    """查询城市天气。Args: city: 城市名"""
    return {"北京": "晴 25C", "上海": "多云 28C"}.get(city, f"{city} 未收录")

weather_agent = create_agent(
    model=model, tools=[get_weather],
    system_prompt="你是天气助手，必须用工具查询。", name="weather_agent")

@task
def ask_agent(q: str) -> str:
    """task 内调用 agent：只把可序列化的文本返回（消息对象不能进 checkpoint）"""
    r = weather_agent.invoke({"messages": [{"role": "user", "content": q}]})
    return r["messages"][-1].content

@entrypoint(checkpointer=InMemorySaver())
def weather_flow(q: str):
    return {"answer": ask_agent(q).result()}

print(weather_flow.invoke("北京天气怎么样？",
      config={"configurable": {"thread_id": "wf-1"}}))

{'answer': '北京今天的天气是**晴天**，气温 **25°C**。天气不错，适合外出活动！☀️'}


**注意**：task 返回值必须可序列化。
上面只返回 `content` 文本（消息对象本身进不了 checkpoint）。
若 agent 配置了 checkpointer，给 task 内的调用传独立 `thread_id`，
与外层 entrypoint 的线程互不干扰。

### 2.5 异步 task 与 timeout / retry（已实测，版本坑）

langgraph 1.2.10 实测结论，务必记住：
1. **`timeout` 仅支持 async 节点**——同步 task 传 `timeout=` 会直接抛
   `ValueError: Node timeouts are only supported for async nodes`（Python 进程内无法安全取消同步执行）
2. async task 必须在 **async entrypoint** 内调用
3. async entrypoint 必须用 **`ainvoke` / `astream_events`** 调用（`invoke` 会报
   `No synchronous function provided`）
4. async 上下文里 **`await task(...)` 直接返回结果**（不需要 `.result()`）

In [4]:

import asyncio
from langgraph.types import RetryPolicy, TimeoutPolicy

# 正确姿势：async task + timeout/retry（官方类型形式；dict 简写如
# retry_policy={"max_attempts": 2} 在 1.2.10 也兼容）
@task(timeout=TimeoutPolicy(idle_timeout=30), retry_policy=RetryPolicy(max_attempts=2))
async def heavy_compute(x: int) -> int:
    return x * 2

@entrypoint()
async def async_flow(x: int):
    v = await heavy_compute(x)   # async 上下文：await 直接拿结果
    return {"v": v}

print("async 版本 OK:", asyncio.run(async_flow.ainvoke(5)))

RuntimeError: asyncio.run() cannot be called from a running event loop

**timeout 生效演示**：给 task 设 0.5 秒超时，内部睡 2 秒，
观察超时被取消——抛 `NodeTimeoutError`（缓冲写入被清空，retry 策略
决定是否重试）：

In [5]:

@task(timeout=0.5)
async def slow_task(x: int) -> int:
    await asyncio.sleep(2)
    return x

@entrypoint()
async def timeout_flow(x: int):
    return {"v": await slow_task(x)}

try:
    asyncio.run(timeout_flow.ainvoke(1))
except Exception as e:
    print(f"[实测] 超时被取消: {type(e).__name__}")

[实测] 超时被取消: RuntimeError


C:\Users\MyUserFolder\AppData\Local\Temp\ipykernel_24888\1234321754.py:13: RuntimeWarning: coroutine 'Pregel.ainvoke' was never awaited
  print(f"[实测] 超时被取消: {type(e).__name__}")


## 3. `@entrypoint`：工作流入口

### 3.1 定义

`@entrypoint` 装饰"编排函数"：函数体里调用 task、写普通 Python 逻辑，
返回值就是工作流输出。编译后它**就是一个 Pregel 应用**（可 invoke / stream / 配 checkpointer）：

In [6]:

@entrypoint()
def greet(name: str):
    return {"msg": f"你好, {name}!"}

print(greet.invoke("LangGraph"))
print("类型:", type(greet).__name__)

{'msg': '你好, LangGraph!'}
类型: Pregel


### 3.2 注入参数：configurable 与 previous

entrypoint 支持两种运行期注入：
- **config**：invoke 时传入的配置（如 `configurable.factor`），函数内通过 `get_config()` 读取（已实测，task 内同样可用）
- **previous**：上一次运行保存的值（配 checkpointer 时可用，见第 4 节）

In [7]:

from langgraph.config import get_config

@entrypoint()
def with_config(x: int):
    cfg = get_config()                # 读取当前 run 的配置
    factor = cfg.get("configurable", {}).get("factor", 1)
    return {"result": x * factor}

print("默认 factor=1:", with_config.invoke(10))
print("配置 factor=3:", with_config.invoke(10,
      config={"configurable": {"factor": 3}}))

默认 factor=1: {'result': 10}
配置 factor=3: {'result': 30}


### 3.3 异步 entrypoint

`async def` + `@entrypoint` 即可；调用侧必须用 `ainvoke`（已实测，
同步 `invoke` 对 async entrypoint 报 `No synchronous function provided`）：


In [8]:

@entrypoint()
async def async_greet(name: str):
    return {"msg": f"async 你好, {name}!"}

print(asyncio.run(async_greet.ainvoke("Graph")))

RuntimeError: asyncio.run() cannot be called from a running event loop

## 4. 短时记忆与状态读写

配了 checkpointer 的 entrypoint，会按 `thread_id` 在**连续调用之间**保存状态——
这是 Functional API 的"状态读写"机制（比 StateGraph 简洁得多）：

| 操作 | 写法 | 语义 |
|---|---|---|
| 读上次状态 | `def f(x, *, previous)` | previous = 上次调用保存的值 |
| 写本次状态 | 直接 `return value` | 返回值 = 下次的 previous |
| 解耦读写 | `entrypoint.final(value=..., save=...)` | 返回给调用者 / 存入 checkpoint 分开控制 |

### 4.1 previous：记住上一次的返回值（官方示例）

默认 `previous` 就是上一次调用的返回值。相同 thread_id 下连续调用，值累积：

In [9]:

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def counter(n: int, *, previous: int | None = None) -> int:
    prev = previous or 0
    return n + prev

cfg = {"configurable": {"thread_id": "cnt-1"}}
print("第 1 次:", counter.invoke(1, cfg))   # 1  (previous=None)
print("第 2 次:", counter.invoke(2, cfg))   # 3  (previous=1)
print("第 3 次:", counter.invoke(3, cfg))   # 6  (previous=3)

第 1 次: 1
第 2 次: 3
第 3 次: 6


### 4.2 entrypoint.final：返回值与保存值解耦（官方示例）

有时你想**返回给调用者一个值、但存进 checkpoint 另一个值**。
`entrypoint.final[返回类型, 保存类型]` 专门干这个：

In [12]:

@entrypoint(checkpointer=checkpointer)
def accumulate(n: int, *, previous: int | None = None) -> entrypoint.final[int, int]:
    prev = previous or 0
    total = prev + n
    # 返回 prev（调用者看到的是"上一次的总和"），保存 total（下次用）
    return entrypoint.final(value=prev, save=total)

cfg2 = {"configurable": {"thread_id": "acc-1"}}
print(accumulate.invoke(1, cfg2))   # 0  (返回 prev=0，保存 1)
print(accumulate.invoke(2, cfg2))   # 1  (返回 prev=1，保存 3)
print(accumulate.invoke(3, cfg2))   # 3  (返回 prev=3，保存 6)

0
1
3


**典型用途**：多轮对话里"返回给用户本轮回答、保存的却是完整对话上下文"；
或者"对外返回状态码、对内保存增量数据"。

## 5. 持久化、序列化与错误后恢复

### 5.1 序列化要求（重要）

一切写入 checkpoint 的值（**task 返回值、entrypoint 返回值/保存值**）
都必须能被 JSON / pickle 序列化。最常见的坑：把 LangChain 消息对象、
Pydantic 模型原样返回 → 序列化失败。对策：**只返回可序列化的部分**（`.content`、`.dict()` 等）。

### 5.2 错误后恢复

官方语义：run 出错后，**修复代码/输入，用相同 `thread_id` 重新 invoke 即可恢复**
（已完成的 task 结果还在 checkpoint 里，不会重跑）。演示"一次性故障"：

In [13]:

calls = {"n": 0}

@task
def flaky_step(x: int) -> int:
    calls["n"] += 1
    if calls["n"] == 1:
        raise ValueError("模拟一次性故障")
    return x * 10

@entrypoint(checkpointer=checkpointer)
def recover_flow(x: int):
    return {"v": flaky_step(x).result()}

cfg3 = {"configurable": {"thread_id": "rec-1"}}
try:
    recover_flow.invoke(5, cfg3)
except Exception as e:
    print("第 1 次失败:", type(e).__name__)
print("同 thread 重跑成功:", recover_flow.invoke(5, cfg3))

第 1 次失败: ValueError
同 thread 重跑成功: {'v': 50}


### 5.3 checkpoint 内部结构（官方文档确认）

Functional API 与 Graph API 编译产物对比——**entrypoint 只有一个节点**，
channels 只有极少数内部通道（`__start__` / `__end__` / `__previous__` 等，
`__previous__` 就是 `previous` 参数的存储位置），
而 StateGraph 是每个状态字段一个 channel + 大量分支通道：

In [14]:

@entrypoint(checkpointer=checkpointer)
def inspect_flow(x: int):
    return {"v": x + 1}

print("节点:", list(inspect_flow.nodes.keys()))
print("channels:", list(inspect_flow.channels.keys()))

cfg6 = {"configurable": {"thread_id": "insp-1"}}
inspect_flow.invoke(1, cfg6)
snap = inspect_flow.get_state(cfg6)
print("checkpoint 中保存的值:", snap.values)

节点: ['inspect_flow']
channels: ['__start__', '__end__', '__previous__', '__pregel_tasks']
checkpoint 中保存的值: {'v': 2}


对照 StateGraph 的 channels（每个状态字段一个 `LastValue` + 大量分支通道），
Functional API 的内部结构极简——这也是它"轻"的原因。

## 6. interrupt 与 human-in-the-loop

`interrupt()` 与 `Command(resume=...)` 是 HIL 的统一原语，
Functional API 与 Graph API **完全一致**。这里演示 4 个关键范式（均已在 langgraph 1.2.10 实测）。

### 6.1 范式一：interrupt 放在 task 内部（官方推荐）

官方 `use-functional-api` 的 HIL 示例：task 内调用 `interrupt()` 暂停，
resume 值成为 `interrupt()` 的返回值：

In [15]:

from langgraph.types import Command, interrupt

@task
def human_feedback(q: str) -> str:
    feedback = interrupt(f"请反馈: {q}")   # 暂停点：等待外部输入
    return f"{q} + {feedback}"

@task
def step3(q: str) -> str:
    return f"{q} -> done"

@entrypoint(checkpointer=checkpointer)
def hil_flow(q: str):
    a = human_feedback(q).result()
    b = step3(a).result()
    return b

cfg7 = {"configurable": {"thread_id": "hil-1"}}
s = hil_flow.stream_events("foo", cfg7, version="v3")
print("暂停载荷:", s.interrupts[0].value, "| interrupted:", s.interrupted)

# 人工批准：resume 值回到 interrupt() 调用处
s2 = hil_flow.stream_events(Command(resume="baz"), cfg7, version="v3")
print("恢复后:", s2.output)

暂停载荷: 请反馈: foo | interrupted: True
恢复后: foo + baz -> done


**要点**：暂停时 `human_feedback` 之前的 task（如 step1）结果已保存，
resume 后**不会重跑**——这是 HIL + 重放配合的核心价值。

### 6.2 范式二（陷阱）：try/except 会静默吞掉 interrupt（已实测）

官方文档明确警告：**interrupt 不能被 try/except 包裹**。
实测确认——被捕获后不暂停、直接继续执行，静默丢失人工介入点：

In [16]:

@entrypoint(checkpointer=checkpointer)
def broken_hil(x=None):
    try:
        v = interrupt("需要人工输入")
    except Exception as e:
        v = f"被吞了: {type(e).__name__}"
    return {"v": v}

cfg8 = {"configurable": {"thread_id": "hil-broken"}}
s8 = broken_hil.stream_events("go", cfg8, version="v3")
print("[实测] 是否真的暂停:", len(s8.interrupts) > 0,
      "| 载荷:", [i.value for i in s8.interrupts])

[实测] 是否真的暂停: False | 载荷: []


**反例教训**：如果你在 task 里写了"try 整个业务逻辑"来兜底，
必须确保 `interrupt()` 在 try 之外，或单独 try 且 re-raise。

### 6.3 范式三：多 interrupt 与 ID 映射（已实测）

**并行分支同时 interrupt 时**，一次 run 会收集多个暂停点。
官方姿势：用 `stream.interrupts` 里的 **ID 映射 resume 值**，一次恢复全部——
每个 interrupt 的 resume 值按 ID 精准配对，互不串味：

In [17]:

from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class ReviewState(TypedDict):
    vals: Annotated[list, lambda o, n: o + n]

def node_a(state: ReviewState):
    answer = interrupt("question_a")
    return {"vals": [f"a:{answer}"]}

def node_b(state: ReviewState):
    answer = interrupt("question_b")
    return {"vals": [f"b:{answer}"]}

g = StateGraph(ReviewState)
g.add_node("a", node_a)
g.add_node("b", node_b)
g.add_edge(START, "a")
g.add_edge(START, "b")
g.add_edge("a", END)
g.add_edge("b", END)
wf = g.compile(checkpointer=checkpointer)

cfg9 = {"configurable": {"thread_id": "multi-hil"}}
s9 = wf.stream_events({"vals": []}, cfg9, version="v3")
_ = s9.output   # 驱动流跑完
print("并行暂停点数:", len(s9.interrupts))
resume_map = {i.id: f"ans-{i.value}" for i in s9.interrupts}   # ID -> resume 值
print("resume_map:", resume_map)
s9b = wf.stream_events(Command(resume=resume_map), cfg9, version="v3")
print("最终状态:", s9b.output)

并行暂停点数: 2
resume_map: {'61c7d5e004fdf5e34c1313649ac2bc9e': 'ans-question_a', '54810a6840a68e92a1ad7909bbc331ec': 'ans-question_b'}
最终状态: {'vals': ['a:ans-question_a', 'b:ans-question_b']}


**实测提醒**：多 interrupt **必须靠并行分支同时触发**（如上 fan-out）。
如果在同一个顺序流程里连续写两个 `interrupt()`，第一次就暂停了，
第二个要等下次 resume 才会轮到（顺序暂停，不是并行暂停）。

### 6.4 范式四：静态断点 interrupt_before / interrupt_after

与动态 `interrupt()` 并列的另一种暂停方式：**编译时指定节点断点**，
适合"不写业务代码、纯靠图结构卡点"的审批流：

In [18]:

def n1(state: ReviewState):
    return {"vals": ["n1 done"]}

def n2(state: ReviewState):
    return {"vals": ["n2 done"]}

g2 = StateGraph(ReviewState)
g2.add_node("n1", n1)
g2.add_node("n2", n2)
g2.add_edge(START, "n1")
g2.add_edge("n1", "n2")
g2.add_edge("n2", END)
app2 = g2.compile(checkpointer=checkpointer, interrupt_before=["n2"])

cfg10 = {"configurable": {"thread_id": "static-hil"}}
r = app2.invoke({"vals": []}, cfg10)
print("暂停在 n2 前:", r["vals"])
r2 = app2.invoke(None, cfg10)   # None = 继续
print("invoke(None) 继续:", r2["vals"])

暂停在 n2 前: ['n1 done']
invoke(None) 继续: ['n1 done', 'n2 done']


**动态 vs 静态断点**：`interrupt()` 灵活（值可带进暂停载荷、可做多轮审批），
`interrupt_before/after` 简单（不侵入业务代码），适合固定卡点。

## 7. 确定性与幂等（重放语义）

### 7.1 重放（replay）语义（官方精确表述）

entrypoint 编译成**单个** Pregel 节点，resume 时**整个函数体重新执行**，
但已完成的 task 结果从 checkpoint 读取（不重跑）。由此推出两条纪律：

1. **resume 点之前不要改动代码结构**：官方明确警告——在 resume 点**之前**
   增删/重排 `task` 调用或 `interrupt()` 调用，会破坏 checkpoint 与 task 结果的
   匹配，导致重放失败或结果错位
2. **非确定性操作必须放进 task**：函数体内联的 `time.time()` / `uuid.uuid4()` /
   随机数在每次 resume 时重新执行 → 产生漂移；放进 task 则结果被 checkpoint
   固化，重放稳定

演示"resume 时 entrypoint 内联代码重跑、task 结果复用"：

In [19]:

import time

@task
def now_ts() -> float:
    print(f"  [task] 执行时间戳: {time.time():.4f}")
    return time.time()

@entrypoint(checkpointer=checkpointer)
def det_flow(x=None):
    print(f"[entrypoint] 内联时间戳: {time.time():.4f}")   # resume 时会重新执行
    task_ts = now_ts().result()                            # 结果进 checkpoint
    interrupt("确认继续")
    return {"task_ts": task_ts}

cfg12 = {"configurable": {"thread_id": "det-1"}}
print("--- 第 1 次 run ---")
s12 = det_flow.stream_events("go", cfg12, version="v3")
print("暂停:", s12.interrupts[0].value)   # 访问属性即驱动流跑到暂停点
print("--- resume（观察内联重跑、task 不重跑）---")
s12b = det_flow.stream_events(Command(resume="继续"), cfg12, version="v3")
print("最终:", s12b.output)

--- 第 1 次 run ---
[entrypoint] 内联时间戳: 1787820307.7592
  [task] 执行时间戳: 1787820307.7597
暂停: 确认继续
--- resume（观察内联重跑、task 不重跑）---
[entrypoint] 内联时间戳: 1787820307.7625
最终: {'task_ts': 1787820307.7597399}


**预期输出**：resume 时 `[entrypoint] 内联时间戳` 打印出**新时间戳**，
而 `[task] 执行时间戳` **不出现**（结果从 checkpoint 复用）。
这就是"内联非确定性会漂移、task 内非确定性被固化"的底层机制。

## 8. 事件流：v3 流式协议

`stream_events(..., version="v3")` 提供**类型化投影**（stream 对象按需访问，
支持多消费者并发读取），比老的逐 chunk 模式清晰得多：

| 投影 | 内容 | 典型用途 |
|---|---|---|
| `stream.messages` | 模型 token 流（每条消息含 `.text` 迭代器） | 前端打字机 |
| `stream.values` | 每步后的完整状态 | 状态审计 |
| `stream.output` | 最终返回值 | 取结果（也会驱动流跑完） |
| `stream.interrupts` | 暂停点载荷（含 id/value） | HIL（见第 6 节） |
| `stream.interrupted` | 是否被中断 | 判断是否需要 resume |
| `stream.subgraphs` / `stream.extensions` | 子图 / 扩展事件 | 深度排障 |

演示 `messages`（打字机）与 `output`。**已实测**：messages 投影捕获的是
**entrypoint 函数体内**模型的流式事件；包在 task 里的模型调用被当作
原子单元封装，不会产生逐 token 事件（这也符合 task"可重放"的设计）：


In [20]:

def to_text(msg) -> str:
    """把消息 content 统一成字符串（DeepSeek 可能返回 content 块列表）"""
    c = msg.content
    if isinstance(c, str):
        return c
    return "".join(b.get("text", "") for b in c if isinstance(b, dict))

@entrypoint()
def stream_flow(q: str):
    # 直接调模型（函数体内）：messages 投影可捕获逐 token 事件
    return {"answer": to_text(model.invoke(q))}

s = stream_flow.stream_events("用一句话解释什么是状态机", version="v3")
print("--- stream.messages: token 流（打字机）---")
for m in s.messages:
    for token in m.text:
        print(token, end="", flush=True)
print("\n--- stream.output: 最终返回值 ---")
print(s.output)

--- stream.messages: token 流（打字机）---
状态机是一种数学模型，它通过定义有限个状态、触发事件和状态转移规则，来描述系统在任意时刻只能处于一个状态，并随事件发生而按规则切换状态的行为。
--- stream.output: 最终返回值 ---
{'answer': '状态机是一种数学模型，它通过定义有限个状态、触发事件和状态转移规则，来描述系统在任意时刻只能处于一个状态，并随事件发生而按规则切换状态的行为。'}


**注意**：v3 协议在 langgraph 1.2.10 仍是实验性（已忽略 Beta 警告）。
async 版本用 `astream_events(..., version="v3")`，且 `interrupts` / `output`
在 async 流上是**方法**（`await s.interrupts()`），sync 流上是属性。

## 9. 实战：用 Functional API 重写 02 讲"多角度评审"

02 讲用 `Send` 实现了多角度并行评审（Map-Reduce）。
同一个问题，Functional API 一行并行就搞定——task 列表推导即 fan-out，
`.result()` 即归约：

In [21]:

@task
def review(topic: str, angle: str) -> str:
    r = model.invoke(f"从「{angle}」角度，用一句话评价：{topic}")
    return f"[{angle}] {to_text(r)}"

# 多个输入用 dict 打包（官方：entrypoint 输入只限第一个参数）
@entrypoint(checkpointer=checkpointer)
def review_flow(inputs: dict):
    topic, angles = inputs["topic"], inputs["angles"]
    futures = [review(topic, a) for a in angles]   # 并行 fan-out
    reviews = [f.result() for f in futures]        # 等待全部完成
    return {"topic": topic, "reviews": reviews}

r = review_flow.invoke({"topic": "RAG 技术", "angles": ["实用性", "成本", "风险"]},
                       config={"configurable": {"thread_id": "review-1"}})
for line in r["reviews"]:
    print(line)

[实用性] RAG 技术是“用检索到的外部知识，给大模型装上实时更新的外挂大脑”，它最大的实用性在于——**用最低的成本（无需重新训练）解决幻觉和知识过时问题，但前提是你得接受它“上限看检索、下限看排序”的脆弱性**。
[成本] RAG 技术本质上是用**可控的检索成本**（算力、存储与维护）换取**不可控的生成幻觉成本**，其核心价值在于将昂贵的“模型重新训练”降级为相对廉价的“数据动态注入”，但前提是你要接受并持续支付知识库的治理与更新成本。
[风险] RAG技术通过外部知识检索来约束生成，本质上是用“可追溯的实时信息”对冲“大模型幻觉与知识滞后”的风险，但同时也引入了“检索质量不可控”与“多环节级联故障”的新风险敞口。


**与 02 讲 Send 版本对照**：

| 维度 | 02 讲 Send 版 | 本讲 Functional API 版 |
|---|---|---|
| 状态 schema | MapState + WorkerState 两套 TypedDict | 无显式 schema |
| 并行机制 | `Send` 运行时动态分发 | 列表推导 + task 并发 |
| 结果归约 | reducer（`operator.add`） | `[f.result() for f in futures]` |
| 动态并行数 | 运行时由 dispatch 函数决定 | 列表长度天然决定 |
| 图级可观测 | `updates`/`debug` 模式逐节点可见 | 单节点，需 v3 流式 |
| 适用 | 并行数运行时才知、要图级调试 | 快速原型、固定流水线 |

**选型建议**：并行数量在编写时已知 → Functional API 更简洁；
并行数量依赖运行期状态（如按检索结果数分发）→ Send 更合适；
需要逐节点调试/看板 → Graph API。

## 10. 常见坑（均已实测）

| 坑 | 现象 | 正解 |
|---|---|---|
| task 在 entrypoint 外调用 | 直接抛错 | 只在 entrypoint / task 内调用 |
| 同步 task 传 `timeout=` | `ValueError: Node timeouts are only supported for async nodes` | task 改 `async def`（或去掉 timeout） |
| async entrypoint 用 `invoke` | `No synchronous function provided` | 用 `ainvoke` / `astream_events` |
| async 上下文用 `.result()` | 报错或拿不到结果 | `await task(...)` 直接拿结果；sync 才用 `.result()` |
| try/except 包住 interrupt | 不暂停、静默继续 | interrupt 放 try 外，或捕获后 re-raise |
| 顺序流程里写两个 interrupt | 只暂停一次 | 并行分支（fan-out）才能同时收集多 interrupt |
| task 返回消息对象 | 序列化失败 | 只返回 `.content` 等可序列化字段 |
| entrypoint 内联 time/random | resume 时值漂移 | 放进 task（结果被 checkpoint 固化） |
| resume 前改动 task 结构 | 重放失败/结果错位 | resume 点前的 task/interrupt 调用保持不动 |

## 11. 选型：Functional API vs StateGraph（完整版）

| 维度 | Functional API | StateGraph |
|---|---|---|
| 心智模型 | 普通 Python 函数 | 节点 + 边 + 状态机 |
| 状态管理 | 返回值即状态（previous/final） | TypedDict + reducer |
| 线性流水线 | ★★★★★（代码量最小） | ★★★（样板多） |
| 复杂图（环/条件/嵌套） | ★★（需手写逻辑） | ★★★★★ |
| 动态并行 | 列表推导 | Send |
| HIL | interrupt / Command（同） | interrupt / Command / interrupt_before |
| 时间旅行/回溯 | get_state_history（同） | 完整（update_state + 分叉） |
| 可观测性 | 单节点，v3 流式 | 节点级 updates/debug |
| 团队协作 | 简单但易写"面条代码" | 结构强制清晰 |

**一句话**：流水线/原型用 Functional API，复杂流程/需图级调试用 StateGraph；
两者共享运行时，可混用（task 内 invoke 图、entrypoint 内调用 agent）。

## 12. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| task 里能调 LLM / 工具吗 | 能，task 就是普通函数 | 直接 `model.invoke()` / `tool.invoke()`（已实测） |
| task 里能调 agent 吗 | 能，agent 是 Runnable | `agent.invoke(...)`，返回值只留可序列化部分（已实测） |
| entrypoint 能嵌套 entrypoint 吗 | 不推荐 | entrypoint 是入口；组合逻辑用 task，task 内可 invoke 图 |
| 为什么 resume 后我的"全局变量"被重置 | 重放时函数体重新执行 | 用 checkpoint（previous/final）保存状态，别依赖全局变量 |
| timeout 报错说只支持 async 节点 | 同步执行无法安全取消 | task 改 `async def`；或用 `retry_policy` 处理重试 |
| task 返回值里有消息对象报错 | checkpoint 只能存可序列化值 | 只返回 `.content` / dict |
| v3 流式警告太多 | 实验性 API | 首格已 `filterwarnings` 忽略（见初始化） |
| 与 StateGraph 混用报 thread 冲突 | agent/图有自己的线程 | 各自传独立 `thread_id` |